<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [3]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

</br>

# 학습 내용
>이번 장에서는 <strong>Linear Probing(선형 프로빙)</strong>에 대해 학습합니다.</br></br>
>사전학습 모델의 특성 추출 능력을 활용하여 분류 헤드만 학습하는 전이학습 기법을 학습해봅시다.

</br>

# 전이학습 (Transfer Learning)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">사전학습된 모델의 지식을 새로운 태스크에 재활용</mark>하는 기법입니다.
> 적은 데이터와 짧은 학습 시간으로 높은 성능을 달성할 수 있습니다.

이미지 분류 모델을 처음부터 학습하려면 수백만 장의 레이블된 이미지와 수일에서 수주의 GPU 학습 시간이 필요합니다. 대부분의 실무  상황에서는 이런 자원이 없습니다. ImageNet(1,400만 장)으로 사전학습된 모델은 이미 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">에지, 질감, 형태, 객체 부분</mark> 등 다양한 시각 특징을 추출하는 방법을 알고 있으므로, 이 지식을  새로운 태스크에 재활용하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">소량의 데이터와 짧은 학습 시간</mark>으로 높은 성능을 달성할 수 있습니다.</br>이러한 전이학습의 가장 간단한 형태가 **Linear Probing**입니다. 사전학습 모 델의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">백본(Backbone) 전체를 동결(Freeze)</mark>하고, 마지막 분류 레이어(fc)만 교체하여 새로 학습합니다. 학습 파라미터가 전체의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.05% 미만</mark>이므로 학습이 매우 빠르고, 데이터가 적을 때 과적합 위험이 낮으며, 사전학습 특징이 새 태스크에도 유효한지 빠르게 검증할 수 있습니다.</br>이 장을 학습하기 위해서는 신경망 기본 구조(레이어, 가중치, 순전파), CNN(합성곱 신경망)의 역할, 그리고 PyTorch에서 역전파 여부를 제어하는 `requires_grad` 개념에 대한 이해가 필요합니다.

</br>

## Linear Probing
> 사전학습 모델의 가중치를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">완전히 동결(Freeze)</mark>하고, 마지막 분류층(fc)만 교체하여 학습합니다.

In [ ]:
# TODO 1: ResNet18 사전학습 모델을 model에 로드해봅시다. (weights=models.ResNet18_Weights.IMAGENET1K_V1)

import torchvision.models as models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
print(model)

KeyboardInterrupt: 

In [ ]:
# TODO 2: 모든 파라미터를 동결(requires_grad=False)해봅시다.

for param in model.parameters():
    param.requires_grad = False

print(f"동결 완료: 모든 파라미터의 requires_grad = False")

In [ ]:
# TODO 3: fc 레이어를 10개 클래스에 맞게 교체해봅시다.

model.fc = nn.Linear(model.fc.in_features, 10)
print(f"새 fc 레이어: {model.fc}")

In [ ]:
# TODO 4: 전체/학습 가능 파라미터 수를 출력해봅시다.

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체: {total_params:,}, 학습 가능: {trainable_params:,} ({trainable_params/total_params:.1%})")

💡왜 동결하는가?
> 사전학습된 특성 추출기는 이미 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">범용적인 시각 특성을 학습</mark>했습니다.</br>
> 적은 데이터로 전체를 학습하면 과적합 위험이 크므로, 헤드만 학습하는 것이 안전합니다.

</br>

## 이미지 전처리 (transforms)
> 사전학습 모델이 학습한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 전처리</mark>를 적용해야 합니다.

In [ ]:
# TODO 5: ImageNet 사전학습 모델에 맞는 전처리 파이프라인을 구성해봅시다.

# TODO 5-1: Resize(256)과 CenterCrop(224)을 적용해봅시다.
# TODO 5-2: ToTensor()로 텐서 변환해봅시다.
# TODO 5-3: ImageNet 평균/표준편차로 Normalize해봅시다.
#            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print(f"전처리 파이프라인: {transform}")

💡Normalize 값
> ImageNet 사전학습 모델을 사용할 때는 반드시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 통계값</mark>으로 정규화해야 합니다.</br>
> 다른 값을 사용하면 성능이 크게 하락합니다.

## ResNet18 구조 요약

</br>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">conv1</td><td>7x7 합성곱 + BatchNorm + ReLU</td></tr>
    <tr><td style="text-align:center">layer1~4</td><td>Residual Block (잔차 연결)</td></tr>
    <tr><td style="text-align:center">avgpool</td><td>Global Average Pooling</td></tr>
    <tr><td style="text-align:center">fc</td><td>완전 연결층 (분류 헤드)</td></tr>
  </tbody>
</table>

💡Linear Probing vs Fine-tuning
> Linear Probing: 헤드만 학습 → <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">빠르고 안전</mark>, 데이터 적을 때</br>
> Fine-tuning: 전체 모델 학습 → 성능 높지만 과적합 위험, 데이터 충분할 때

</br>

## 데이터 준비 (CIFAR-10)
> CIFAR-10 데이터셋을 로드하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 사전학습 모델에 맞는 전처리</mark>를 적용합니다.</br>
> 위에서 정의한 `transform`을 사용하여 이미지를 변환합니다.

In [ ]:
# TODO 6: cifar10_train에 CIFAR-10 학습 데이터를 로드해봅시다. (transform 적용, download=True)

cifar10_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
print(f"학습 데이터: {len(cifar10_train)}개")
print(f"클래스: {cifar10_train.classes}")

In [ ]:
# TODO 7: cifar10_test에 CIFAR-10 테스트 데이터를 로드해봅시다.

cifar10_test = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
print(f"테스트 데이터: {len(cifar10_test)}개")

In [ ]:
# TODO 8: train_loader를 생성해봅시다. (batch_size=256, shuffle=True)

train_loader = DataLoader(cifar10_train, batch_size=256, shuffle=True, num_workers=2)
print(f"train_loader: {len(train_loader)} 배치")

In [ ]:
# TODO 9: test_loader를 생성해봅시다. (batch_size=256, shuffle=False)

test_loader = DataLoader(cifar10_test, batch_size=256, shuffle=False, num_workers=2)
print(f"test_loader: {len(test_loader)} 배치")

</br>

## 모델 학습 (Training Loop)
> 손실 함수, 옵티마이저, 스케줄러를 정의하고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Linear Probing 방식으로 학습</mark>합니다.</br>
> `model.fc.parameters()`만 옵티마이저에 전달하여 분류 헤드만 학습합니다.

In [ ]:
# TODO 10: criterion에 CrossEntropyLoss를 저장해봅시다.

criterion = nn.CrossEntropyLoss()
print(f"손실 함수: {criterion}")

In [ ]:
# TODO 11: optimizer에 SGD를 저장해봅시다. (model.fc.parameters(), lr=0.001, momentum=0.9)

optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
print(f"옵티마이저: {optimizer}")

In [ ]:
# TODO 12: scheduler에 StepLR을 저장해봅시다. (step_size=2, gamma=0.1)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)
print(f"스케줄러: {scheduler}")

In [ ]:
# TODO 13: 학습 루프를 구현해봅시다.

model = model.to(device)
num_epochs = 5
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # TODO 13-1: 기울기를 초기화해봅시다.
        optimizer.zero_grad()

        # TODO 13-2: 순전파를 수행하고 outputs에 저장해봅시다.
        outputs = model(images)

        # TODO 13-3: 손실을 계산해봅시다.
        loss = criterion(outputs, labels)

        # TODO 13-4: 역전파를 수행해봅시다.
        loss.backward()

        # TODO 13-5: 파라미터를 업데이트해봅시다.
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # TODO 13-6: 에폭 종료 후 스케줄러를 업데이트해봅시다.
    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total
    train_losses.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%")

💡StepLR 스케줄러
> `StepLR`은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">step_size 에폭마다 학습률을 gamma 배로 감소</mark>시킵니다.</br>
> 예: `step_size=2, gamma=0.1`이면 2 에폭마다 학습률이 1/10로 줄어듭니다.</br>
> 학습 초반에는 큰 학습률로 빠르게 수렴하고, 후반에는 작은 학습률로 세밀하게 조정합니다.

</br>

## 모델 평가 (Evaluation)
> 학습된 모델을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">테스트 데이터셋으로 평가</mark>하여 Linear Probing의 성능을 확인합니다.

In [ ]:
# TODO 14: 테스트 데이터로 모델을 평가해봅시다.

# TODO 14-1: 모델을 평가 모드로 전환해봅시다.
model.eval()

correct = 0
total = 0

# TODO 14-2: torch.no_grad() 블록 안에서 테스트 데이터를 순회하며 정확도를 계산해봅시다.
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_accuracy = 100.0 * correct / total
print(f"Linear Probing 테스트 정확도: {test_accuracy:.2f}%")

💡model.eval()의 역할
> `model.eval()`은 모델을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">평가 모드</mark>로 전환합니다.</br>
> Dropout이 비활성화되고, BatchNorm은 학습 중 계산된 running statistics를 사용합니다.</br>
> 추론 시에는 반드시 `torch.no_grad()`와 함께 사용하여 불필요한 그래디언트 계산을 방지합니다.